# Modelo de recompensa, baselines e políticas adaptativas

O modelo de recompensa é uma regressão logística calibrada, ajustada somente no período de treino e condicionada ao contexto e ao canal histórico. O canal entra como ação candidata na estimativa de recompensa, não como atributo pré-decisão. O baseline fixo escolhe o canal com maior conversão observada no treino; Thompson Sampling e Epsilon-Greedy usam as estimativas contextuais para selecionar ações. As taxas históricas são observacionais e não identificam, por si só, efeito causal do canal.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next(path for path in candidates if (path / 'src' / 'adaptive_offers').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from adaptive_offers.baselines import stats_table
from adaptive_offers.config import DEFAULT_DATA_PATH
from adaptive_offers.data import load_bank_marketing, validate_schema
from adaptive_offers.policies import EpsilonGreedyPolicy, FixedBestPolicy, Guardrails, ThompsonSamplingPolicy
from adaptive_offers.preparation import temporal_split
from adaptive_offers.reward_model import ChannelRewardModel

df = load_bank_marketing(DEFAULT_DATA_PATH)
validate_schema(df)
splits = temporal_split(df)
train, validation, test = splits.train, splits.validation, splits.test
display(stats_table(train))
fixed = FixedBestPolicy().fit(train)
print('Melhor ação no treino:', fixed.best)

## Modelo de recompensa calibrado por canal

In [ ]:
reward_model = ChannelRewardModel.fit(train)
metrics = pd.DataFrame({
    "validation": reward_model.evaluate_observed(validation),
    "test": reward_model.evaluate_observed(test),
})
display(metrics)
print('Ações:', reward_model.actions)

## Thompson Sampling e Epsilon-Greedy

In [ ]:
actions = reward_model.actions
thompson = ThompsonSamplingPolicy(
    actions, seed=42, guardrails=Guardrails(), reward_model=reward_model,
).fit(train)
epsilon = EpsilonGreedyPolicy(
    actions, epsilon=0.10, seed=42, guardrails=Guardrails(), reward_model=reward_model,
).fit(train)
example = validation.iloc[0].drop(labels=['y', 'contact', 'duration'])
expected = reward_model.predict_all(pd.DataFrame([example.to_dict()])).iloc[0].to_dict()
print('Recompensa esperada por canal, dado o contexto:', expected)
print('Baseline fixo:', fixed.best)
for name, policy in [('thompson_sampling', thompson), ('epsilon_greedy', epsilon)]:
    action = policy.select_action(example, request_id=f'notebook-{name}')
    probabilities = {item: policy.action_probability(example, item) for item in actions}
    print(name, {
        'action': action,
        'probabilities': probabilities,
        'policy_version': policy.policy_version,
    })